# 05 - Prediction Outputs

## Goal
In this notebook, we generate interpretable prediction outputs from the selected Random Forest model.

The goal is to create a business-facing dataset that includes:
- actual no-show outcomes,
- predicted no-show probabilities,
- risk classifications under different thresholds.

This output will later support dashboard development and decision-oriented interpretation.

In [1]:
import pandas as pd
import joblib

In [2]:
df = pd.read_csv("../data/medical_no_show_preprocessed.csv")
df.head()

,specialty,gender,no_show,disability,city,appointment_month,appointment_year,appointment_shift,age,under_12_years_old,...,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity,appointment_day_of_week,appointment_hour,age_missing,weather_missing
0,physiotherapy,M,1,Missing,Missing,sept,2021,afternoon,NaN,0,...,23.7,0.2,1,1,no_rain,mild,Thursday,13,True,False
1,psychotherapy,M,0,Missing,Missing,sept,2021,afternoon,NaN,0,...,23.7,0.2,1,1,no_rain,mild,Thursday,13,True,False
2,speech therapy,F,0,Missing,Missing,sept,2021,afternoon,NaN,0,...,23.7,0.2,1,1,no_rain,mild,Thursday,13,True,False
3,physiotherapy,F,0,Missing,Missing,sept,2021,afternoon,NaN,0,...,23.7,0.2,1,1,no_rain,mild,Thursday,13,True,False
4,physiotherapy,M,0,motor,B. CAMBORIU,sept,2021,afternoon,68.0,0,...,23.7,0.2,1,1,no_rain,mild,Thursday,14,False,False


In [3]:
random_forest_model = joblib.load(
    "../models/random_forest_no_show_pipeline.joblib"
)

In [4]:
X = df.drop(columns=["no_show"])
y = df["no_show"]

In [5]:
no_show_probability = random_forest_model.predict_proba(X)[:, 1]

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
test_no_show_probability = random_forest_model.predict_proba(X_test)[:, 1]

In [8]:
prediction_outputs = X_test.copy()

prediction_outputs["actual_no_show"] = y_test.values
prediction_outputs["no_show_probability"] = test_no_show_probability

prediction_outputs["predicted_no_show_threshold_0_25"] = (
    prediction_outputs["no_show_probability"] >= 0.25
).astype(int)

prediction_outputs["predicted_no_show_threshold_0_30"] = (
    prediction_outputs["no_show_probability"] >= 0.30
).astype(int)

prediction_outputs.head()

,specialty,gender,disability,city,appointment_month,appointment_year,appointment_shift,age,under_12_years_old,over_60_years_old,...,rain_intensity,heat_intensity,appointment_day_of_week,appointment_hour,age_missing,weather_missing,actual_no_show,no_show_probability,predicted_no_show_threshold_0_25,predicted_no_show_threshold_0_30
32492,Missing,F,intellectual,CAMBORIU,april,2019,morning,NaN,0,0,...,no_rain,warm,Wednesday,8,True,False,0,0.095851,0,0
13078,occupational therapy,F,intellectual,CAMBORIU,april,2017,afternoon,16.0,0,0,...,no_rain,mild,Friday,17,False,False,0,0.133333,0,0
30950,psychotherapy,M,intellectual,B. CAMBORIU,mar,2019,afternoon,7.0,1,0,...,moderate,warm,Monday,15,False,False,0,0.233333,0,0
47086,Missing,M,intellectual,B. CAMBORIU,dec,2016,afternoon,12.0,1,0,...,no_rain,warm,Tuesday,16,False,False,0,0.036075,0,0
16003,speech therapy,M,intellectual,ITAJAÍ,july,2017,afternoon,12.0,1,0,...,no_rain,mild,Wednesday,14,False,False,0,0.000000,0,0


In [9]:
def risk_level(probability):
    if probability >= 0.30:
        return "High Risk"
    elif probability >= 0.25:
        return "Moderate Risk"
    else:
        return "Low Risk"

prediction_outputs["risk_level"] = prediction_outputs[
    "no_show_probability"
].apply(risk_level)

In [10]:
prediction_outputs[
    [
        "actual_no_show",
        "no_show_probability",
        "predicted_no_show_threshold_0_25",
        "predicted_no_show_threshold_0_30",
        "risk_level"
    ]
].head(10)

,actual_no_show,no_show_probability,predicted_no_show_threshold_0_25,predicted_no_show_threshold_0_30,risk_level
32492,0,0.095851,0,0,Low Risk
13078,0,0.133333,0,0,Low Risk
30950,0,0.233333,0,0,Low Risk
47086,0,0.036075,0,0,Low Risk
16003,0,0.000000,0,0,Low Risk
48707,0,0.273333,1,0,Moderate Risk
5172,0,0.113333,0,0,Low Risk
40156,0,0.003333,0,0,Low Risk
4520,0,0.100000,0,0,Low Risk
731,0,0.020000,0,0,Low Risk


In [11]:
risk_level_summary = (
    prediction_outputs.groupby("risk_level")["actual_no_show"]
    .agg(
        total_appointments="count",
        actual_no_show_count="sum",
        actual_no_show_rate=lambda x: x.mean() * 100
    )
)

risk_level_summary["actual_no_show_rate"] = (
    risk_level_summary["actual_no_show_rate"].round(2)
)

risk_level_summary

,total_appointments,actual_no_show_count,actual_no_show_rate
risk_level,,,
High Risk,952,389,40.86
Low Risk,8764,526,6.00
Moderate Risk,203,51,25.12


### Risk Segmentation Insight

The model-generated risk groups show a clear separation in actual no-show behavior:

- **Low Risk:** 6.00% actual no-show rate
- **Moderate Risk:** 25.12% actual no-show rate
- **High Risk:** 40.86% actual no-show rate

This indicates that the selected model is not only producing probabilities, but also creating practically meaningful risk segments.  
Appointments classified as **High Risk** are substantially more likely to result in a no-show than Low Risk appointments.

In [12]:
prediction_outputs.to_csv(
    "../outputs/no_show_prediction_outputs.csv",
    index=False
)

In [13]:
risk_level_summary.to_csv(
    "../outputs/risk_level_summary.csv"
)

In [14]:
dashboard_outputs = prediction_outputs.copy()

dashboard_outputs["actual_appointment_outcome"] = (
    dashboard_outputs["actual_no_show"]
    .map({0: "Attended", 1: "No-Show"})
)

dashboard_outputs["flagged_at_threshold_0_25"] = (
    dashboard_outputs["predicted_no_show_threshold_0_25"]
    .map({0: "Not Flagged", 1: "Flagged"})
)

dashboard_outputs["flagged_at_threshold_0_30"] = (
    dashboard_outputs["predicted_no_show_threshold_0_30"]
    .map({0: "Not Flagged", 1: "Flagged"})
)

dashboard_outputs["no_show_probability_percent"] = (
    dashboard_outputs["no_show_probability"] * 100
).round(2)

dashboard_outputs.head()

,specialty,gender,disability,city,appointment_month,appointment_year,appointment_shift,age,under_12_years_old,over_60_years_old,...,weather_missing,actual_no_show,no_show_probability,predicted_no_show_threshold_0_25,predicted_no_show_threshold_0_30,risk_level,actual_appointment_outcome,flagged_at_threshold_0_25,flagged_at_threshold_0_30,no_show_probability_percent
32492,Missing,F,intellectual,CAMBORIU,april,2019,morning,NaN,0,0,...,False,0,0.095851,0,0,Low Risk,Attended,Not Flagged,Not Flagged,9.59
13078,occupational therapy,F,intellectual,CAMBORIU,april,2017,afternoon,16.0,0,0,...,False,0,0.133333,0,0,Low Risk,Attended,Not Flagged,Not Flagged,13.33
30950,psychotherapy,M,intellectual,B. CAMBORIU,mar,2019,afternoon,7.0,1,0,...,False,0,0.233333,0,0,Low Risk,Attended,Not Flagged,Not Flagged,23.33
47086,Missing,M,intellectual,B. CAMBORIU,dec,2016,afternoon,12.0,1,0,...,False,0,0.036075,0,0,Low Risk,Attended,Not Flagged,Not Flagged,3.61
16003,speech therapy,M,intellectual,ITAJAÍ,july,2017,afternoon,12.0,1,0,...,False,0,0.000000,0,0,Low Risk,Attended,Not Flagged,Not Flagged,0.00


In [15]:
dashboard_outputs.to_csv(
    "../outputs/no_show_dashboard_dataset.csv",
    index=False
)